In [1]:
# import pandas as pd
# import numpy as np
# from tqdm import tqdm
# from sklearn.preprocessing import StandardScaler
# from sklearn.base import BaseEstimator, TransformerMixin

# class ShiftedLogScaler(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         self.shift_ = np.abs(np.min(X))
#         X_shifted = np.log1p(X + self.shift_)
#         self.mean_ = np.mean(X_shifted)
#         self.std_ = np.std(X_shifted)
#         return self

#     def transform(self, X):
#         X_clipped = np.clip(X, -self.shift_, None)
#         return (np.log1p(X_clipped + self.shift_) - self.mean_) / (self.std_ + 1e-8)

#     def inverse_transform(self, X):
#         return np.expm1((X * self.std_) + self.mean_) - self.shift_

# class TabularReachDataset:
#     def __init__(self, samples, scalers):
#         self.samples = samples
#         self.scalers = scalers

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         sample = self.samples[idx]
#         return {
#             "svs_idx": sample["svs_idx"],
#             "reach_id": sample["reach_id"],
#             "x_d": sample["x_d"],
#             "d_mask": sample["d_mask"],
#             "x_s": sample["x_s"],
#             "hws": sample["hws"],
#             "y": sample["y"],
#         }

#     def denormalize(self, x_scaled_arr, col_name):
#         scaler = self.scalers.get(col_name)
#         if scaler is None:
#             return x_scaled_arr
            
#         original_shape = x_scaled_arr.shape
#         x_inv = scaler.inverse_transform(x_scaled_arr.reshape(-1, 1))
#         return x_inv.reshape(original_shape)

# def _read_list(filepath):
#     with open(filepath, "r", encoding="utf-8") as f:
#         return [int(line.strip()) for line in f if line.strip()] 

# def build_tabular_datasets(cfg):
#     dynamic_df = pd.read_parquet(cfg.parquet_file)
#     static_df = pd.read_parquet(cfg.attributes_file)

#     sites_dict = {
#         subset: _read_list(getattr(cfg, f"{subset}_basin_file"))
#         for subset in ['train', 'validate', 'test']
#     }

#     dynamic_df['raw_h'] = dynamic_df['d_wse']
#     dynamic_df['raw_w'] = dynamic_df['width']
#     dynamic_df['raw_s'] = dynamic_df['slope']
    
#     train_dynamic_df = dynamic_df[dynamic_df['reach_id'].isin(sites_dict['train'])]
#     train_static_df = static_df[static_df['reach_id'].isin(sites_dict['train'])]
    
#     dynamic_cols = [ff for f in cfg.features.dynamic.values() for ff in f]
#     dyn_cols_to_scale = dynamic_cols + cfg.features.target
#     static_cols_to_scale = cfg.features.static 
    
#     scalers = {}

#     def scale_features(full_df, train_df, columns):
#         full_df = full_df.copy()
#         protected_cols = {'reach_id', 'svs_idx', 'raw_h', 'raw_w', 'raw_s'}
        
#         for col in columns:
#             if col in protected_cols or col in getattr(cfg, 'no_norm_cols', []):
#                 continue
                
#             train_data = train_df[[col]].values 
#             full_data = full_df[[col]].values
                
#             if col in getattr(cfg, 'log_norm_cols', []):
#                 scaler = ShiftedLogScaler()
#             else:
#                 scaler = StandardScaler()
                
#             scaler.fit(train_data)
#             full_df[col] = scaler.transform(full_data).flatten()
#             scalers[col] = scaler
                
#         return full_df

#     dynamic_df = scale_features(dynamic_df, train_dynamic_df, dyn_cols_to_scale)
#     static_df = scale_features(static_df, train_static_df, static_cols_to_scale)

#     static_df = static_df.set_index('reach_id')

#     splits = {}
#     for subset, sites in sites_dict.items():
#         splits[subset] = []
#         subset_df = dynamic_df[dynamic_df['reach_id'].isin(sites)]
        
#         for site_id, site_df in tqdm(subset_df.groupby('reach_id'), disable=cfg.quiet):
#             n_samples = len(site_df)
#             if n_samples < 10: 
#                 continue
#             if n_samples > cfg.pad_size:
#                 raise ValueError(f"Location {site_id} has {n_samples} samples, exceeding {cfg.pad_size=}")

#             hws_arr = site_df[['raw_h', 'raw_w', 'raw_s']].values.astype(np.float32)
            
#             if cfg.hw_corr_thresh is not None:
#                 corr_matrix = np.corrcoef(hws_arr[:,0], hws_arr[:,1])
#                 if np.isnan(corr_matrix[0, 1]) or corr_matrix[0, 1] <= cfg.hw_corr_thresh:
#                     continue 
            
#             x_d_arr = site_df[dynamic_cols].values.astype(np.float32)
#             y_arr = site_df[cfg.features.target].values.astype(np.float32)
#             svs_idx = site_df['svs_idx'].iloc[0] 

#             x_s_arr = static_df.loc[site_id, static_cols_to_scale].values.astype(np.float32)

#             if cfg.median_w:
#                 hws_arr[:, 1] = np.median(hws_arr[:, 1])
#             if cfg.median_s:
#                 hws_arr[:, 2] = np.median(hws_arr[:, 2])

#             x_pad = np.ones((cfg.pad_size, len(dynamic_cols)), dtype=np.float32)
#             x_pad[:n_samples, :] = x_d_arr

#             y_pad = np.ones((cfg.pad_size, len(cfg.features.target)), dtype=np.float32)
#             y_pad[:n_samples, :] = y_arr

#             hws_pad = np.ones((cfg.pad_size, 3), dtype=np.float32)
#             hws_pad[:n_samples, :] = hws_arr

#             mask = np.zeros((cfg.pad_size, len(dynamic_cols)), dtype=np.int8) 
#             mask[:n_samples, :] = 1

#             splits[subset].append({
#                 "svs_idx": svs_idx,
#                 "reach_id": site_id,
#                 "x_d": x_pad,
#                 "d_mask": mask,
#                 "x_s": x_s_arr,
#                 "hws": hws_pad,
#                 "y": y_pad,
#             })

#     return {
#         subset: TabularReachDataset(split, scalers)
#         for subset, split in splits.items()
#     }

In [2]:
from config import Config
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/mcfli/swot_vD.yml")
cfg.quiet = False
cfg.num_workers = 4

In [3]:
from data import get_dataloaders

dls = get_dataloaders(cfg)

100%|██████████| 193/193 [00:00<00:00, 642.88it/s]


In [4]:
for x in dls['train']:
    break

In [5]:
x.x_d

array([[[ 0.13673829,  0.18266965, -0.40139496, ..., -0.17517206,
         -0.31046364, -0.26490688],
        [ 0.13679542,  0.27400678,  1.370848  , ..., -0.17517206,
         -0.21445791, -0.18552892],
        [ 0.13665332,  0.18001395,  0.5384453 , ..., -0.17517206,
         -0.32149452, -0.38293374],
        ...,
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ]],

       [[ 0.13746405, -0.20566006, -0.4558919 , ..., -0.17517206,
         -0.27445433, -0.25958002],
        [ 0.13766858, -0.21153183, -0.42719254, ..., -0.17517206,
         -0.27327633, -0.25216326],
        [ 0.13737288, -0.21757086, -0.7061178 , ..., -0.17517206,
         -0.27408564, -0.262885  ],
        ...,
        [ 1.        ,  1.        ,  1.        , ...,  

In [6]:
x.to_jax().x_d

Array([[[ 0.13673829,  0.18266965, -0.40139496, ..., -0.17517206,
         -0.31046364, -0.26490688],
        [ 0.13679542,  0.27400678,  1.370848  , ..., -0.17517206,
         -0.21445791, -0.18552892],
        [ 0.13665332,  0.18001395,  0.5384453 , ..., -0.17517206,
         -0.32149452, -0.38293374],
        ...,
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ]],

       [[ 0.13746405, -0.20566006, -0.4558919 , ..., -0.17517206,
         -0.27445433, -0.25958002],
        [ 0.13766858, -0.21153183, -0.42719254, ..., -0.17517206,
         -0.27327633, -0.25216326],
        [ 0.13737288, -0.21757086, -0.7061178 , ..., -0.17517206,
         -0.27408564, -0.262885  ],
        ...,
        [ 1.        ,  1.        ,  1.        , ...,  

In [2]:
from data import build_tabular_datasets

In [3]:
datasets = build_tabular_datasets(cfg)

100%|██████████| 193/193 [00:00<00:00, 535.72it/s]


In [ ]:
datasets['train'].scalers

In [48]:
splits

NameError: name 'splits' is not defined

In [6]:
static_df = pd.read_parquet(cfg.attributes_file)
static_df

area_km2   uparea_km2  dis_m3_pyr  dis_m3_pmn  \
basin         subbasin                                                         
EAUF-V7200010 21000002       468.102025  4364.328752   66.342003   22.684999   
              21000020       401.905725  3625.201976   48.854000   16.743000   
              21000022       292.309508  3076.492166   28.339001    9.780000   
              21000026       481.039377  1767.807919    6.031000    2.077000   
              21000032       390.920772   621.770285    8.653000    3.024000   
...                                 ...          ...         ...         ...   
USGS-50129254 USGS-50129254   42.674493    42.674493    4.198000    1.576000   
USGS-50138000 USGS-50136400   51.402873    51.402873    0.185000    0.068000   
              USGS-50138000  261.453879   312.856752    5.788000    2.068000   
USGS-50144000 USGS-50144000  349.557308   349.557308    9.208000    3.183000   
USGS-50147800 USGS-50147800  195.872993   195.872993    5.115000    1.855000   

                             dis_m3_pmx  run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin         subbasin                                                        
EAUF-V7200010 21000002       108.996002         525           0           0   
              21000020        80.602997         537           0           0   
              21000022        46.625000         504           0           0   
              21000026         9.826000         422           0           0   
              21000032        14.126000         437           0           0   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    8.597000         578          61          69   
USGS-50138000 USGS-50136400    0.359000         598          63          79   
              USGS-50138000   11.577000         593          30          48   
USGS-50144000 USGS-50144000   19.839001         609          36          61   
USGS-50147800 USGS-50147800   10.214000         601          26          38   

                             inu_pc_slt  lka_pc_sse  ...  urb_pc_sse  \
basin         subbasin                               ...               
EAUF-V7200010 21000002                1           0  ...           0   
              21000020                0           0  ...           0   
              21000022                0           1  ...           0   
              21000026                0           0  ...           0   
              21000032                0           0  ...           0   
...                                 ...         ...  ...         ...   
USGS-50129254 USGS-50129254          72           1  ...          22   
USGS-50138000 USGS-50136400          80           0  ...          48   
              USGS-50138000          51           0  ...          27   
USGS-50144000 USGS-50144000          62           1  ...           4   
USGS-50147800 USGS-50147800          38           0  ...          39   

                             nli_ix_sav  rdd_mk_sav  hft_ix_s93  hft_ix_s09  \
basin         subbasin                                                        
EAUF-V7200010 21000002              844         711         135         106   
              21000020             1754         786         187         163   
              21000022              842         631         133          88   
              21000026              728         759         142          92   
              21000032              824         536         127          86   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254        3209        2365         296         277   
USGS-50138000 USGS-50136400        4778        5447         363         346   
              USGS-50138000        3680        2432         291         288   
USGS-50144000 USGS-50144000        1686        1292         222         173   
USGS-50147800 USGS-50147800        3856        1703         301         286  

In [32]:
[f for f in cfg.features.dynamic.values()]

[['d_x_area',
  'slope',
  'slope_u',
  'slope_r_u',
  'slope2',
  'slope2_u',
  'slope2_r_u',
  'width',
  'p_width',
  'width_u',
  'wse',
  'wse_u',
  'wse_r_u',
  'p_length',
  'reach_q',
  'reach_q_b',
  'dark_frac',
  'ice_clim_f',
  'partial_f',
  'n_good_nod',
  'obs_frac_n',
  'xovr_cal_q',
  'xtrk_dist',
  'd_wse']]

In [7]:
from config import Config
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/swot_vD.yml")
cfg.quiet = False



In [8]:
getattr(cfg, 'test'+'_basin_file')

PosixPath('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/basin_lists/test_swot.csv')